# `rdat_ndegen_applied` symmetry check

Validates the `write_ndegen_applied` `_r.dat`/`_hr.dat` route against `.mmn`. The goal is a position-matrix source that does not read `.mmn` at all (tens of GB per structure). `wannier.position_matrix`'s `rdat_ndegen_applied` source reads only `_r.dat`/`_hr.dat`/`_wsvec.dat` from a Wannier90 run built with the upstream `write_ndegen_applied` `.win` flag (jaemolihm/wannier90, branch `plan5-write-ndegen-applied` -- see `notes/log/2026-09-13_jaemo_transl_inv_full_ws_distance.md`).

Compares that source against the trusted `mmn` source on ONE Wannierization (same `.chk`, same gauge -- SrTiO3 Ti_Q_2A trial03, 48 WF, n_occ=38), so any difference is purely the position-matrix source, not a different disentanglement. Two checks:

- source-mesh comparison: Tr Omega on the actual ab-initio k-mesh.
- symmetry-enforced covariance: T, C4z, Mx on generic off-grid points, for each source separately.

Set `RUN_COMPUTATION = False` to load the last saved run from `results/rdat_ndegen_applied_symmetry/`.

In [1]:
import json
import sys
from pathlib import Path

import numpy as np

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "validation").exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

from validation._paths import RESULTS, STO_TI_Q_2A_RUN
from validation._paths import DATA
from validation.symmetry.compare_trial_curvature_lib import (
    PREFIX,
    TRIALS,
    build_trial,
    evaluate_trial,
)
from validation.symmetry.check_trial_curvature_geometry_lib import symmetry_checks
from wannier.wannier_io import read_wannier_checkpoint

N_OCC = 38
COMPONENTS = ("yz", "zx", "xy")

## Config

In [2]:
RUN_COMPUTATION = False  # True: rebuild both sources and recompute. False: load results/rdat_ndegen_applied_symmetry/.
DIRECTORY = DATA / "SrTiO3/Ti_Q_2A/output/188914bc889/trial_03_Sr_sp_Ti_pd_O_sp/proj_gauge/191509bc889"
MMN_DIRECTORY = None  # borrow .mmn from another trial on the same NSCF; None -> DIRECTORY
SKIP_MMN = False  # run rdat_ndegen_applied's own symmetry checks only, no mmn_pair reference build
OUTPUT_DIR = RESULTS / "rdat_ndegen_applied_symmetry"
N_GENERIC_POINTS = 24
SEED = 20260915

## Build the trial(s)

`source_results[key]["curvature"]` is the full traced-occupied curvature tensor on the source checkpoint's k-mesh, for each of `"rdat_ndegen_applied"` and (unless `SKIP_MMN`) `"mmn"`.

In [3]:
if RUN_COMPUTATION:
    mmn_directory = MMN_DIRECTORY or DIRECTORY
    include_mmn = not SKIP_MMN
    TRIALS.clear()
    TRIALS.update(
        {
            "rdat_ndegen_applied": {
                "directory": DIRECTORY,
                "source": "rdat_ndegen_applied",
                "label": "rdat_ndegen_applied (_r.dat/_hr.dat, no .mmn)",
            },
        }
    )
    if include_mmn:
        TRIALS["mmn"] = {
            "directory": DIRECTORY,
            "mmn_directory": mmn_directory,
            "source": "mmn",
            "cache": OUTPUT_DIR / "AA_mmn_pair.npz",
            "label": f"mmn_pair (trusted, .chk+.mmn from {mmn_directory})",
        }

    checkpoint = read_wannier_checkpoint(DIRECTORY / f"{PREFIX}.chk")
    source_points = np.asarray(checkpoint["kpt_red"], dtype=float)
    mp_grid = tuple(int(value) for value in checkpoint["mp_grid"])

    rng = np.random.default_rng(SEED)
    generic_points = rng.uniform(0.071, 0.929, size=(N_GENERIC_POINTS, 3))

    keys = ("rdat_ndegen_applied", "mmn") if include_mmn else ("rdat_ndegen_applied",)
    source_results = {}
    symmetries = {}
    for key in keys:
        model, pos = build_trial(key, TRIALS)
        source_results[key] = evaluate_trial(model, pos, source_points)
        symmetries[key] = symmetry_checks(model, pos, generic_points)

rdat_ndegen_applied: 48 WFs, 38 occupied; source=rdat_ndegen_applied; reads _r.dat/_hr.dat (write_ndegen_applied); diagnostics={'ndegen_applied_hermiticity_max': 0.024255080000000012, 'ndegen_applied_hermiticity_relative': 0.007192473993454994, 'centres_max_error': 4.000000020099037e-08, 'zero_H_blocks_added': 0}


    H(R) orbital-dependent mapping (min |R + tau_right - tau_left|): 729 -> 729 R-vectors, 0.000% of the weight moved outside the input set


mmn: 48 WFs, 38 occupied; source=mmn; A(R) Hermiticity=0.000e+00, outside-H(R) weight=0.000e+00


## Compare `rdat_ndegen_applied` against `mmn` on the source mesh

In [4]:
if RUN_COMPUTATION:
    source_mesh_comparison = {
        "point_set": (
            f"all {len(source_points)} source checkpoint nodes on "
            f"{mp_grid[0]}x{mp_grid[1]}x{mp_grid[2]}"
        ),
        "n_occupied": N_OCC,
    }
    if include_mmn:
        left = source_results["mmn"]["trace"]
        right = source_results["rdat_ndegen_applied"]["trace"]
        difference = right - left
        components = {}
        for index, name in enumerate(COMPONENTS):
            reference_l2 = float(np.linalg.norm(left[:, index]))
            components[name] = {
                "mmn_max_abs_A2": float(np.max(np.abs(left[:, index]))),
                "rdat_ndegen_applied_max_abs_A2": float(np.max(np.abs(right[:, index]))),
                "max_abs_difference_A2": float(np.max(np.abs(difference[:, index]))),
                "rms_difference_A2": float(
                    np.sqrt(np.mean(np.abs(difference[:, index]) ** 2))
                ),
                "relative_l2_difference": float(
                    np.linalg.norm(difference[:, index]) / reference_l2
                    if reference_l2 > 0
                    else np.nan
                ),
            }
        source_mesh_comparison["max_abs_energy_difference_eV"] = float(
            np.max(
                np.abs(
                    source_results["rdat_ndegen_applied"]["energies"]
                    - source_results["mmn"]["energies"]
                )
            )
        )
        source_mesh_comparison["components"] = components
    else:
        source_mesh_comparison["note"] = (
            "no mmn reference built (SKIP_MMN); this reports "
            "rdat_ndegen_applied's own symmetry covariance only"
        )

    summary = {
        "directory": str(DIRECTORY),
        "mmn_directory": str(mmn_directory) if include_mmn else None,
        "source_mesh_comparison": source_mesh_comparison,
        "offgrid_symmetry": symmetries,
        "symmetry_conventions": {
            "curvature_vector": "(Tr Omega_yz, Tr Omega_zx, Tr Omega_xy)",
            "spatial_rule": "omega(O k) = det(O) O omega(k)",
            "time_reversal_rule": "omega(-k) = -omega(k)",
            "generic_points": len(generic_points),
        },
    }
    print(json.dumps(summary, indent=2, sort_keys=True))

{
  "directory": "/Users/treycole/Repos/axion-phonon/data/SrTiO3/Ti_Q_2A/output/188914bc889/trial_03_Sr_sp_Ti_pd_O_sp/proj_gauge/191509bc889",
  "mmn_directory": "/Users/treycole/Repos/axion-phonon/data/SrTiO3/Ti_Q_2A/output/188914bc889/trial_03_Sr_sp_Ti_pd_O_sp/proj_gauge/191509bc889",
  "offgrid_symmetry": {
    "mmn": {
      "C4z": {
        "max_abs_A2": 1.5781242979073795e-09,
        "relative_l2": 1.061231582187998e-09,
        "rms_A2": 5.356191716119262e-10
      },
      "Mx": {
        "max_abs_A2": 1.4957053373620965e-09,
        "relative_l2": 1.0836512852073728e-09,
        "rms_A2": 5.469347251259524e-10
      },
      "time_reversal": {
        "max_abs_A2": 3.8029601689970605e-10,
        "relative_l2": 2.6394404257729867e-10,
        "rms_A2": 1.3321643627084524e-10
      }
    },
    "rdat_ndegen_applied": {
      "C4z": {
        "max_abs_A2": 0.06934280909192031,
        "relative_l2": 0.043370726594852284,
        "rms_A2": 0.02116170753968723
      },
      "Mx"

## Save

In [5]:
if RUN_COMPUTATION:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    summary_path = OUTPUT_DIR / "rdat_ndegen_applied_symmetry_summary.json"
    data_path = OUTPUT_DIR / "rdat_ndegen_applied_symmetry.npz"
    summary_path.write_text(json.dumps(summary, indent=2, sort_keys=True) + "\n")
    npz_payload = dict(
        source_kpoints_reduced=source_points,
        generic_kpoints_reduced=generic_points,
        rdat_ndegen_applied_source_trace=source_results["rdat_ndegen_applied"]["trace"],
        rdat_ndegen_applied_source_energies=source_results["rdat_ndegen_applied"]["energies"],
    )
    if include_mmn:
        npz_payload["mmn_source_trace"] = source_results["mmn"]["trace"]
        npz_payload["mmn_source_energies"] = source_results["mmn"]["energies"]
    np.savez_compressed(data_path, **npz_payload)
    print(f"saved {summary_path}")
    print(f"saved {data_path}")

saved /Users/treycole/Repos/axion-phonon/validation/results/rdat_ndegen_applied_symmetry/rdat_ndegen_applied_symmetry_summary.json
saved /Users/treycole/Repos/axion-phonon/validation/results/rdat_ndegen_applied_symmetry/rdat_ndegen_applied_symmetry.npz


## Load previously saved results (skip if you just computed above)

In [6]:
if not RUN_COMPUTATION:
    summary = json.loads((OUTPUT_DIR / "rdat_ndegen_applied_symmetry_summary.json").read_text())
    data = np.load(OUTPUT_DIR / "rdat_ndegen_applied_symmetry.npz")
    source_results = {"rdat_ndegen_applied": {
        "trace": data["rdat_ndegen_applied_source_trace"],
        "energies": data["rdat_ndegen_applied_source_energies"],
    }}
    if "mmn_source_trace" in data:
        source_results["mmn"] = {"trace": data["mmn_source_trace"], "energies": data["mmn_source_energies"]}
    source_points = data["source_kpoints_reduced"]
    generic_points = data["generic_kpoints_reduced"]
    print(json.dumps(summary, indent=2, sort_keys=True))